> **Forked from** `notebooks/demo_notebooks/autonomous-agent-prediction-beta-demo-agent.ipynb`.
>
> This notebook builds a small, deliberate variant of the demo agent to (a) prove the
> full local build → validate pipeline works end-to-end on this machine, and (b) make
> one concrete, well-motivated improvement rather than a redesign.
>
> **The bug this fixes**: the demo's `feature-engineer` skill imputes missing values in
> categorical/ordinal columns but never encodes them to numeric. Any sklearn estimator
> fit directly on its output (`train_engineered.csv`) would crash on the leftover
> `object` dtype columns (e.g. `cat_2`, `ord_5`). Confirmed against the local
> `data/raw/data/train_XX` datasets: columns like `feature_0`, `feature_8` etc. are
> `object` dtype with string values such as `cat_1`, `ord_3`.
>
> **Why the fix is schema-agnostic**: comparing `DATA.md` across `train_01`, `train_05`,
> `train_09` shows the family does **not** keep a fixed feature-to-type mapping —
> `feature_1` is `ordinal` in one dataset and `count` in another, and the number of
> categorical columns varies (some datasets have none at all). So the fix detects
> ordinal vs. nominal by inspecting each column's **values** (`ord_<n>` vs. anything
> else), never by column name or position — the only approach that generalizes across
> unseen datasets from the same family, which is the actual thing being scored here.
>
> **Second change**: `prompts/system.md` now explicitly instructs the agent to select
> its final submission(s) by internal cross-validation score, not by the visible public
> score — directly implementing the competition's core warning about winner's-curse
> overfitting to the public leaderboard.

## Define the Agent Submission

Files are written under `../submissions/01_fe_categorical_encoding/` (relative to this notebook's directory), following the project convention of never editing `sample_submission/` in place and giving each experiment its own directory.

In [1]:
import os

EXP_DIR = "../submissions/01_fe_categorical_encoding"
os.makedirs(f"{EXP_DIR}/agent/prompts", exist_ok=True)
os.makedirs(f"{EXP_DIR}/agent/tools", exist_ok=True)
os.makedirs(f"{EXP_DIR}/agent/skills/feature-engineer/scripts", exist_ok=True)
os.makedirs(f"{EXP_DIR}/agent/skills/feature-engineer/resources", exist_ok=True)
print(f"Submission directory tree created: {EXP_DIR}/")

Submission directory tree created: ../submissions/01_fe_categorical_encoding/


In [2]:
%%writefile ../submissions/01_fe_categorical_encoding/agent/agent.yaml
name: ml_agent
model: gemini-3.5-flash
instruction: !include prompts/system.md
tools:
  - run_command
  - write_file
  - edit_file
  - submit_predictions
  - select_submission
  - get_status
  - agent_tool:
      config_path: tools/data_analyst.yaml
skills:
  - skills/feature-engineer
generate_content_config:
  temperature: 0.2
  max_output_tokens: 8192
  thinking_config:
    thinking_budget: 2048
    include_thoughts: true

Overwriting ../submissions/01_fe_categorical_encoding/agent/agent.yaml


In [3]:
%%writefile ../submissions/01_fe_categorical_encoding/agent/prompts/system.md
## Workflow
1. **Start by delegating EDA** to the `data_analyst` tool. Ask it to analyze
   the training and test data. This is more efficient than doing EDA yourself.
2. Review the analysis and plan your modeling approach.
3. Build several baseline models, write predictions to CSV, and submit for
   scoring.
4. Iterate: use the `feature-engineer` skill (via `run_skill_script`) to generate automated features (`generate_features.py`), and try different model types.
5. **Keep experimenting until you have used all allowed submissions.**
   Each submission is a chance to try a different approach.
6. Review your submissions and select the best for final scoring.
7. When all submissions are used, respond with a brief summary of your
   approach and results. **Responding without a tool call ends the session.**

## Important
- Each submit_predictions call returns a **submission ID** (e.g., "sub_1").
  Track these — you'll use them to select your final submission(s).
- You can select a limited number of submissions for final scoring. The best
  test-set score among your selections becomes your final score.
- **Public scores reflect only a subset of the test set.** Your final score
  is computed on a different (private) subset. Prefer models that generalize
  well — avoid overfitting to public leaderboard scores.
- **Selection rule: choose your final submission(s) by internal cross-validation
  score, not by the public score shown after `submit_predictions`.** Before each
  submission, compute a k-fold CV score (e.g. 5-fold stratified) on the training
  data for that exact modeling approach, and record `(submission_id, cv_score,
  cv_std, public_score)` together. When it's time to call `select_submission`,
  rank by `cv_score` (and prefer lower `cv_std` to break ties), and only use the
  public score as a sanity check that nothing is badly broken. The public score
  is a noisy subset and chasing it directly is how you overfit to it.
- **Use all of your allowed submissions.** Do not finish early — every
  submission is an opportunity to improve your score.
- **Prioritize simple models and computational efficiency.** Try to ensure your
  tool calls return quickly.
- **Your session ends when you respond with text and no tool call.**
  Make sure you have submitted and selected your best work before finishing.

## Tips
- Check your budget with the `get_status` tool periodically
- Use cross-validation on the training data before submitting to estimate performance
- Handle missing values and categorical features properly
- Try multiple model types (RandomForest, GradientBoosting, SVM, etc.)
- Feature engineering often matters more than model selection

Overwriting ../submissions/01_fe_categorical_encoding/agent/prompts/system.md


In [4]:
%%writefile ../submissions/01_fe_categorical_encoding/agent/prompts/data_analyst.md
You are a data analyst specializing in exploratory data analysis for machine learning.

## Your Role
When called, you receive a request to analyze a dataset. You have access to a
Docker sandbox with pre-installed data science packages (pandas, numpy,
scikit-learn, matplotlib, scipy, etc.).

## Working Directory
- `train.csv`: Training data with features and target column
- `test.csv`: Test data (features only)
- `target_col.txt`: Contains the name of the target column

## What to Analyze
Perform a thorough but efficient EDA. Cover these areas:

1. **Shape & Schema**: Row counts, column names, dtypes.
2. **Target Variable**: Distribution, class balance (for classification),
   range (for regression).
3. **Missing Values**: Which columns have nulls, percentages.
4. **Feature Types**: Numeric vs. categorical, cardinality of categoricals.
5. **Distributions**: Summary statistics, skewness of numeric features.
6. **Correlations**: Top correlations with the target, multicollinearity.
7. **Train/Test Comparison**: Whether feature distributions differ between
   train and test sets (potential data leakage or distribution shift).
8. **Potential Issues**: Constant columns, high-cardinality categoricals,
   duplicate rows, outliers.

## Guidelines
- Be concise. Use tables and bullet points, not prose.
- Run Python scripts to compute statistics — don't guess.
- Prioritize actionable insights that will help model building.
- Do NOT build models or make predictions. Your job is analysis only.
- End with a brief "Recommendations" section suggesting modeling approaches
  based on what you found.

Overwriting ../submissions/01_fe_categorical_encoding/agent/prompts/data_analyst.md


In [5]:
%%writefile ../submissions/01_fe_categorical_encoding/agent/tools/data_analyst.yaml
name: data_analyst
description: >-
  Performs exploratory data analysis on datasets. Examines distributions,
  correlations, missing values, feature types, and potential data quality
  issues. Returns a structured analysis report.
model: gemini-3-flash-preview
instruction: !include ../prompts/data_analyst.md
tools:
  - run_command
  - write_file
generate_content_config:
  temperature: 0.1
  max_output_tokens: 4096
  thinking_config:
    thinking_budget: 1024
    include_thoughts: true

Overwriting ../submissions/01_fe_categorical_encoding/agent/tools/data_analyst.yaml


In [6]:
%%writefile ../submissions/01_fe_categorical_encoding/agent/skills/feature-engineer/SKILL.md
---
name: feature-engineer
description: >-
  Provides a robust Python script for automated feature generation.
  Handles missing value imputation, schema-agnostic ordinal/nominal
  categorical encoding, and numeric aggregations.
---

# Feature Engineer Skill

This skill equips the agent with a pre-packaged Python CLI script for automated feature engineering.

## Available Scripts

### 1. `generate_features.py`
Automatically identifies column types, imputes missing values, encodes
categorical/ordinal columns to numeric, and calculates row mean over the
original numeric columns.

**Column typing is schema-agnostic**: the dataset family does not keep a fixed
feature-to-type mapping (e.g. `feature_1` is `ordinal` in one dataset and
`count` in another). Each non-numeric column is typed by inspecting its
*values*, not its name or position:
- If every observed value matches `ord_<int>`, it's treated as **ordinal** and
  encoded to the trailing integer (preserves order).
- Otherwise it's treated as **nominal** and label-encoded via a fitted
  train-derived mapping (unseen test categories map to `-1`).

**Usage via `run_skill_script`**:
```python
run_skill_script(
    skill_name="feature_engineer",
    script_name="generate_features.py",
    args="--train train.csv --test test.csv --target target",
)
```
**Arguments**:
- `--train`: Path to training CSV (default: `train.csv`).
- `--test`: Path to test CSV (default: `test.csv`).
- `--target`: Name of the target column (default: `target`).
- `--id-col`: Name of the row identifier column to pass through untouched,
  excluded from imputation/encoding/row_mean (default: `row_id`).

**Outputs**: Creates `train_engineered.csv` and `test_engineered.csv`, both
fully numeric except for the identifier column — ready to feed directly into
an sklearn estimator.

---

## Domain Knowledge Resources

### `leakage_checklist.md`
A concise guide on preventing data leakage during feature engineering. You can read it using the `load_skill_resource` tool:
```python
load_skill_resource(
    skill_name="feature_engineer",
    resource_name="leakage_checklist.md",
)
```

Overwriting ../submissions/01_fe_categorical_encoding/agent/skills/feature-engineer/SKILL.md


In [7]:
%%writefile ../submissions/01_fe_categorical_encoding/agent/skills/feature-engineer/scripts/generate_features.py
#!/usr/bin/env python3
"""Robust CLI script for automated feature generation.

Imputes missing values and encodes every non-numeric column to numeric so the
output is directly model-ready, then adds a row-mean feature over the original
numeric columns. Categorical/ordinal typing is inferred from each column's
values (an `ord_<int>` pattern means ordinal, preserving order; anything else
is nominal) rather than from column name or position, because the dataset
family does not keep a fixed feature-to-type mapping across datasets.
"""

import argparse
import os
import re
import sys

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

ORDINAL_RE = re.compile(r"^ord_(\d+)$")


def fit_ordinal_map(train_col: pd.Series) -> dict:
    """Map each observed value to its trailing integer, preserving order."""
    return {v: int(ORDINAL_RE.match(v).group(1)) for v in train_col.unique()}


def fit_nominal_map(train_col: pd.Series) -> dict:
    """Map each observed value to a deterministic integer code."""
    return {v: i for i, v in enumerate(sorted(train_col.unique()))}


def main():
    parser = argparse.ArgumentParser(description="Generate automated ML features.")
    parser.add_argument("--train", type=str, default="train.csv", help="Path to train CSV")
    parser.add_argument("--test", type=str, default="test.csv", help="Path to test CSV")
    parser.add_argument("--target", type=str, default="target", help="Target column name")
    parser.add_argument(
        "--id-col",
        type=str,
        default="row_id",
        help="Identifier column to pass through untouched (default: row_id)",
    )
    args = parser.parse_args()

    print(f"Loading datasets: {args.train}, {args.test}...")
    if not os.path.exists(args.train):
        print(f"Error: Train file '{args.train}' not found.")
        sys.exit(1)
    if not os.path.exists(args.test):
        print(f"Error: Test file '{args.test}' not found.")
        sys.exit(1)

    train_df = pd.read_csv(args.train)
    test_df = pd.read_csv(args.test)

    target_series = None
    if args.target in train_df.columns:
        target_series = train_df[args.target]
        train_df = train_df.drop(columns=[args.target])
    else:
        print(f"Warning: Target column '{args.target}' not found in train_df.")

    # Align columns
    common_cols = [c for c in train_df.columns if c in test_df.columns]
    train_df = train_df[common_cols].copy()
    test_df = test_df[common_cols].copy()

    print(f"Initial shape: train={train_df.shape}, test={test_df.shape}")

    # Set the identifier column aside — never impute/encode/aggregate it.
    id_train = train_df.pop(args.id_col) if args.id_col in train_df.columns else None
    id_test = test_df.pop(args.id_col) if args.id_col in test_df.columns else None

    # Identify column types
    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = train_df.select_dtypes(exclude=[np.number]).columns.tolist()

    # 1. Impute missing values (fit on train, transform test)
    if num_cols:
        print(f"Imputing missing values for {len(num_cols)} numeric columns...")
        num_imputer = SimpleImputer(strategy="median")
        train_df[num_cols] = num_imputer.fit_transform(train_df[num_cols])
        test_df[num_cols] = num_imputer.transform(test_df[num_cols])

    if cat_cols:
        print(f"Imputing missing values for {len(cat_cols)} categorical columns...")
        cat_imputer = SimpleImputer(strategy="most_frequent")
        train_df[cat_cols] = cat_imputer.fit_transform(train_df[cat_cols])
        test_df[cat_cols] = cat_imputer.transform(test_df[cat_cols])

    # 2. Encode categorical/ordinal columns to numeric (schema-agnostic: typed
    # by value pattern, not column name/position — see module docstring).
    for col in cat_cols:
        train_values = train_df[col].astype(str)
        is_ordinal = train_values.map(lambda v: bool(ORDINAL_RE.match(v))).all()
        if is_ordinal:
            mapping = fit_ordinal_map(train_values)
            print(f"  {col}: ordinal, {len(mapping)} levels")
        else:
            mapping = fit_nominal_map(train_values)
            print(f"  {col}: nominal, {len(mapping)} categories")

        train_df[col] = train_values.map(mapping)
        test_values = test_df[col].astype(str)
        test_df[col] = test_values.map(mapping)

        if is_ordinal:
            # Unseen test category: fall back to the median train code.
            fallback = float(np.median(list(mapping.values())))
        else:
            # Unseen test category: reserved sentinel below the observed range.
            fallback = -1
        test_df[col] = test_df[col].fillna(fallback)

    # 3. Aggregation Features (over the *original* numeric columns only)
    if len(num_cols) > 0:
        print("Calculating row-wise mean feature...")
        train_df["row_mean"] = train_df[num_cols].mean(axis=1)
        test_df["row_mean"] = test_df[num_cols].mean(axis=1)

    # Re-attach identifier and target
    if id_train is not None:
        train_df.insert(0, args.id_col, id_train)
        test_df.insert(0, args.id_col, id_test)
    if target_series is not None:
        train_df[args.target] = target_series

    print(f"Engineered shape: train={train_df.shape}, test={test_df.shape}")
    train_df.to_csv("train_engineered.csv", index=False)
    test_df.to_csv("test_engineered.csv", index=False)
    print("Saved train_engineered.csv and test_engineered.csv successfully.")


if __name__ == "__main__":
    main()

Overwriting ../submissions/01_fe_categorical_encoding/agent/skills/feature-engineer/scripts/generate_features.py


In [8]:
%%writefile ../submissions/01_fe_categorical_encoding/agent/skills/feature-engineer/resources/leakage_checklist.md
# Data Leakage Prevention Checklist

Data leakage occurs when information from outside the training dataset is used to create the model, leading to overly optimistic performance estimates during local validation and catastrophic failure on the private leaderboard.

When performing feature engineering, strictly adhere to the following principles:

## Target Leakage Prevention
- **Rule**: Ensure no feature is directly derived from or highly correlated with the target column in a way that would not be available at true inference time.

Overwriting ../submissions/01_fe_categorical_encoding/agent/skills/feature-engineer/resources/leakage_checklist.md


## Sanity-check the feature engineering fix

Run the new `generate_features.py` directly against one of the local training datasets (bypassing the agent loop, which needs an LLM key) to confirm the output is fully numeric and an sklearn estimator can actually fit on it — this is the concrete bug the fork fixes.

In [9]:
import subprocess
import sys

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

DATASET = "../data/train_01"
SCRIPT = f"{EXP_DIR}/agent/skills/feature-engineer/scripts/generate_features.py"

result = subprocess.run(
    [
        sys.executable,
        SCRIPT,
        "--train", f"{DATASET}/train.csv",
        "--test", f"{DATASET}/test.csv",
        "--target", "target",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("generate_features.py failed")

train_eng = pd.read_csv("train_engineered.csv")
test_eng = pd.read_csv("test_engineered.csv")
print("dtypes after encoding:")
print(train_eng.drop(columns=["row_id", "target"]).dtypes.value_counts())
assert train_eng.drop(columns=["row_id", "target"]).select_dtypes(exclude="number").empty, (
    "Non-numeric columns remain — encoding fix did not work"
)

X = train_eng.drop(columns=["row_id", "target"])
y = train_eng["target"]
clf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=1)
scores = cross_val_score(clf, X, y, cv=5, scoring="roc_auc", n_jobs=1)
print(f"5-fold CV ROC-AUC on train_01 with engineered features: {scores.mean():.4f} +/- {scores.std():.4f}")

Loading datasets: ../data/train_01/train.csv, ../data/train_01/test.csv...
Initial shape: train=(14957, 13), test=(10000, 13)
Imputing missing values for 7 numeric columns...
Imputing missing values for 5 categorical columns...
  feature_0: nominal, 4 categories
  feature_4: nominal, 7 categories
  feature_6: nominal, 3 categories
  feature_8: ordinal, 7 levels
  feature_11: nominal, 7 categories
Calculating row-wise mean feature...
Engineered shape: train=(14957, 15), test=(10000, 14)
Saved train_engineered.csv and test_engineered.csv successfully.

dtypes after encoding:
float64    8
int64      5
Name: count, dtype: int64


5-fold CV ROC-AUC on train_01 with engineered features: 0.6978 +/- 0.0059


## Package and validate the submission

Zip the experiment directory into `submission.zip` at the repo root, then run the project's `validate_submission.py` against it — this is the "is it working" check available without an LLM API key or container runtime (both still pending; see `docs/memory.md`).

In [10]:
import zipfile
from pathlib import Path

exp_path = Path("../submissions/01_fe_categorical_encoding/agent")
zip_path = Path("../submissions/01_fe_categorical_encoding/submission.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in exp_path.rglob("*"):
        if file_path.is_file() and ".ipynb_checkpoints" not in file_path.parts:
            zf.write(file_path, file_path.relative_to(exp_path))

print(f"Wrote {zip_path.resolve()}")

Wrote /home/giova/workspace/autonomous-agent-prediction/submissions/01_fe_categorical_encoding/submission.zip


In [11]:
import subprocess

result = subprocess.run(
    [
        "../.venv/bin/python",
        "../validate_submission.py",
        "--agent-dir", "submissions/01_fe_categorical_encoding/agent",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Validation failed"


=== Pre-flight Validation: submissions/01_fe_categorical_encoding/agent ===

[Check 1] Found agent.yaml at /home/giova/workspace/autonomous-agent-prediction/submissions/01_fe_categorical_encoding/agent/agent.yaml
[Check 2] YAML syntax and !include prompt files are valid.
[Check 3] Required top-level keys present for LlmAgent (Name: ml_agent).
[Check 4] All requested models are fully valid and permitted in models.yaml.
[Check 5] Performing dry-run compilation of ADK agent...
[Success] Agent 'ml_agent' compiled successfully with 7 tools.

>>> VALIDATION SUCCESSFUL: Submission is ready for Kaggle upload! <<<


/home/giova/workspace/autonomous-agent-prediction/.venv/lib/python3.12/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.SKILL_TOOLSET is enabled.
  check_feature_enabled()



## Status

- ✅ Notebook builds `submissions/01_fe_categorical_encoding/` and packages `submission.zip`.
- ✅ `generate_features.py`'s encoding fix verified directly against `train_01`: output is
  fully numeric and a `RandomForestClassifier` fits and cross-validates on it without error
  (the un-forked version would have raised on the leftover `object` columns).
- ✅ `validate_submission.py` passes — the agent config is structurally valid and compiles.
- ⏸️ **Not yet run**: a full `run_local_eval.py` pass (the actual agent loop, driven by a
  live LLM inside the sandbox container). That still needs a configured `data/raw/.env`
  (LLM API key) and a container runtime (`podman`/`docker`), neither of which is set up on
  this machine yet — tracked in `docs/memory.md`.